In [1]:
import pandas as pd
import requests
import folium

print("Everything works!")

Everything works!


In [2]:
import requests
import pandas as pd
import os
import urllib.request

In [3]:
BASE_URL = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
PACKAGE_ID = "major-crime-indicators"

pkg = requests.get(
    f"{BASE_URL}/api/3/action/package_show",
    params={"id": PACKAGE_ID}
).json()

resources = pkg["result"]["resources"]

for r in resources:
    print(r["name"], "|", r["format"], "|", r["url"])

major-crime-indicators.csv | CSV | https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/6909ea60-ef0e-465f-b5d0-43106b6b9130/resource/8084057c-b2da-486c-b517-38f4e314b820/download/major-crime-indicators.csv
major-crime-indicators.xml | XML | https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/6909ea60-ef0e-465f-b5d0-43106b6b9130/resource/29d85821-e756-4970-b1f1-beadb5e62567/download/major-crime-indicators.xml
major-crime-indicators.json | JSON | https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/6909ea60-ef0e-465f-b5d0-43106b6b9130/resource/0f15c1f7-491f-44f8-b278-91c65520d6a4/download/major-crime-indicators.json
Community Safety Indicators | JSON | https://ckan0.cf.opendata.inter.prod-toronto.ca/datastore/dump/11581817-b148-4dd4-99c4-679649515ccc
Community Safety Indicators.csv | CSV | https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/6909ea60-ef0e-465f-b5d0-43106b6b9130/resource/174089c6-5a1a-42a5-8937-f9f25c72700a/download/community-safety-indicators.csv
Community 

In [4]:
resource = next(
    r for r in resources
    if r["format"].upper() == "CSV"
)

print("Resource name:", resource["name"])
print("CSV URL:", resource["url"])

Resource name: major-crime-indicators.csv
CSV URL: https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/6909ea60-ef0e-465f-b5d0-43106b6b9130/resource/8084057c-b2da-486c-b517-38f4e314b820/download/major-crime-indicators.csv


In [5]:
import os
os.makedirs("/content/data", exist_ok=True)
os.makedirs("/content/output", exist_ok=True)

csv_url = resource["url"]
save_path = "/content/data/mci_toronto.csv"

if not os.path.exists(save_path):
    print("Downloading file...")
    urllib.request.urlretrieve(csv_url, save_path)
    print("Download complete!")
else:
    print("File already exists.")

FileNotFoundError: [Errno 2] No such file or directory: '../data/mci_toronto.csv'

In [ ]:
df = pd.read_csv("/content/data/mci_toronto.csv", low_memory=False)
print(df.shape)
df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
df = df.dropna(subset=["LAT_WGS84", "LONG_WGS84"])
df["OCC_YEAR"] = pd.to_numeric(df["OCC_YEAR"], errors="coerce")
df_recent = df[df["OCC_YEAR"] >= 2020].copy()

print("Total rows with coordinates:", len(df))
print("Rows from 2020 onward:", len(df_recent))
print(df_recent["MCI_CATEGORY"].value_counts())

In [ ]:
sample = df_recent.sample(3000, random_state=42)

In [ ]:
import folium
from folium.plugins import MarkerCluster

colour_map = {
    "Assault": "red",
    "Break and Enter": "orange",
    "Robbery": "purple",
    "Auto Theft": "darkblue",
    "Theft Over": "green"
}

m = folium.Map(
    location=[43.6532, -79.3832],
    zoom_start=11,
    tiles="CartoDB positron"
)

cluster = MarkerCluster(name="Crime Incidents").add_to(m)

In [ ]:
for _, row in sample.iterrows():
    crime = row["MCI_CATEGORY"]
    color = colour_map.get(crime, "gray")

    folium.CircleMarker(
        location=[row["LAT_WGS84"], row["LONG_WGS84"]],
        radius=4,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>{crime}</b><br>"
            f"Offence: {row.get('OFFENCE','N/A')}<br>"
            f"Date: {row.get('OCC_DATE','N/A')}<br>"
            f"Neighbourhood: {row.get('NEIGHBOURHOOD_158','N/A')}",
            max_width=250
        )
    ).add_to(cluster)

In [ ]:
folium.LayerControl().add_to(m)

m

In [ ]:
m.save("/content/output/crime_cluster_map.html")
print("Saved!")

In [ ]:
heat_data = df_recent[["LAT_WGS84", "LONG_WGS84"]].dropna().values.tolist()

print("Total points for heatmap:", len(heat_data))

In [ ]:
from folium.plugins import HeatMap

m2 = folium.Map(
    location=[43.6532, -79.3832],
    zoom_start=11,
    tiles="CartoDB dark_matter"
)

In [ ]:
HeatMap(
    heat_data,
    radius=10,
    blur=15,
    min_opacity=0.3,
    gradient={
        0.4: "blue",
        0.65: "lime",
        1: "red"
    }
).add_to(m2)

In [ ]:
m2

In [ ]:
m2.save("/content/output/crime_heatmap.html")
print("Heatmap saved!")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1️⃣ Crime type distribution
crime_counts = df_recent["MCI_CATEGORY"].value_counts()
axes[0].bar(crime_counts.index, crime_counts.values)
axes[0].set_title("Crime Type Distribution")
axes[0].tick_params(axis="x", rotation=30)

# 2️⃣ Yearly trend
yearly = df_recent.groupby("OCC_YEAR").size()
axes[1].plot(yearly.index, yearly.values, marker="o")
axes[1].set_title("Yearly Trend")
axes[1].set_xlabel("Year")

# 3️⃣ Monthly seasonality
df_recent["OCC_MONTH"] = pd.to_datetime(df_recent["OCC_DATE"], errors="coerce").dt.month
monthly = df_recent.groupby("OCC_MONTH").size()
axes[2].bar(monthly.index, monthly.values)
axes[2].set_title("Monthly Seasonality")
axes[2].set_xlabel("Month")

plt.tight_layout()

In [ ]:
plt.savefig("/content/output/crime_stats.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
neigh_counts = df_recent.groupby("NEIGHBOURHOOD_158").size().reset_index(name="count")

print(neigh_counts.head())

In [ ]:
top_neigh = neigh_counts.sort_values(by="count", ascending=False)

print(top_neigh.head(10))

In [ ]:
pivot = df_recent.groupby(["NEIGHBOURHOOD_158", "OCC_YEAR"]).size().unstack()

pivot.head()

In [ ]:
import numpy as np

neigh_counts["log_crime"] = np.log1p(neigh_counts["count"])

neigh_counts.corr()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(neigh_counts["count"], bins=30)
plt.title("Crime Distribution by Neighbourhood")
plt.show()